# 05 — Error Analysis

**DRISHTI** — Investigate false positives and false negatives before polishing

"Look at your false positives first — they'll mostly be rock clusters."
This notebook builds FP/FN galleries, per-class error breakdowns, and the
confusion matrix heatmap that tells you exactly where the model struggles.

**Run on:** Colab/Kaggle with GPU.

In [ ]:
!pip install -q ultralytics matplotlib numpy opencv-python-headless seaborn

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict
from ultralytics import YOLO

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100

CLASSES = ['bottle','can','chain','drink_carton','hook','propeller',
           'shampoo_bottle','standing_bottle','tire','valve','wall','net']

MODEL_PATH = 'drishti_training/runs/yolov8s_seg_baseline/weights/best.pt'
TEST_IMAGES = Path('drishti_training/data/splits/test/images')
TEST_LABELS = Path('drishti_training/data/splits/test/labels')

model = YOLO(MODEL_PATH)
print(f'Model loaded. Test images: {len(list(TEST_IMAGES.glob("*.png")))}')

## 1. Collect All Predictions + Ground Truth

In [ ]:
def parse_gt(label_path, img_w, img_h):
    boxes = []
    if not label_path.exists(): return boxes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5: continue
            cls_id = int(parts[0])
            coords = list(map(float, parts[1:]))
            xs, ys = coords[0::2], coords[1::2]
            boxes.append({'cls': cls_id, 'x1': min(xs)*img_w, 'y1': min(ys)*img_h,
                          'x2': max(xs)*img_w, 'y2': max(ys)*img_h, 'matched': False})
    return boxes

def compute_iou(b1, b2):
    ix1, iy1 = max(b1['x1'],b2['x1']), max(b1['y1'],b2['y1'])
    ix2, iy2 = min(b1['x2'],b2['x2']), min(b1['y2'],b2['y2'])
    if ix2<=ix1 or iy2<=iy1: return 0
    inter = (ix2-ix1)*(iy2-iy1)
    a1 = (b1['x2']-b1['x1'])*(b1['y2']-b1['y1'])
    a2 = (b2['x2']-b2['x1'])*(b2['y2']-b2['y1'])
    return inter/(a1+a2-inter+1e-10)

# Collect
all_fp = []  # false positives: (img_path, pred_box, conf, pred_cls)
all_fn = []  # false negatives: (img_path, gt_box, gt_cls)
all_tp = []  # true positives
confusion = np.zeros((len(CLASSES)+1, len(CLASSES)+1), dtype=int)  # +1 for background
IOU_THRESH = 0.5

for img_path in sorted(TEST_IMAGES.glob('*.png')):
    img = cv2.imread(str(img_path))
    ih, iw = img.shape[:2]
    gt_boxes = parse_gt(TEST_LABELS / f'{img_path.stem}.txt', iw, ih)
    
    results = model.predict(str(img_path), conf=0.25, verbose=False)
    if not results or results[0].boxes is None:
        # All GT boxes are FN
        for gt in gt_boxes:
            all_fn.append((str(img_path), gt, gt['cls']))
            confusion[gt['cls'], -1] += 1  # predicted as background
        continue
    
    pred_boxes = []
    for i in range(len(results[0].boxes)):
        box = results[0].boxes.xyxy[i].cpu().numpy()
        pred_boxes.append({
            'cls': int(results[0].boxes.cls[i]),
            'conf': float(results[0].boxes.conf[i]),
            'x1': box[0], 'y1': box[1], 'x2': box[2], 'y2': box[3],
            'matched': False
        })
    
    # Match predictions to GT
    pred_boxes.sort(key=lambda x: x['conf'], reverse=True)
    
    for pred in pred_boxes:
        best_iou, best_gt = 0, None
        for gt in gt_boxes:
            if gt['matched']: continue
            iou = compute_iou(pred, gt)
            if iou > best_iou:
                best_iou = iou; best_gt = gt
        
        if best_iou >= IOU_THRESH and best_gt is not None:
            best_gt['matched'] = True
            pred['matched'] = True
            confusion[best_gt['cls'], pred['cls']] += 1
            if best_gt['cls'] == pred['cls']:
                all_tp.append((str(img_path), pred, pred['cls']))
            else:
                all_fp.append((str(img_path), pred, pred['cls']))  # wrong class
        else:
            all_fp.append((str(img_path), pred, pred['cls']))
            confusion[-1, pred['cls']] += 1  # background → pred class
    
    for gt in gt_boxes:
        if not gt['matched']:
            all_fn.append((str(img_path), gt, gt['cls']))
            confusion[gt['cls'], -1] += 1

print(f'True Positives:  {len(all_tp)}')
print(f'False Positives: {len(all_fp)}')
print(f'False Negatives: {len(all_fn)}')
precision = len(all_tp) / max(len(all_tp) + len(all_fp), 1)
recall = len(all_tp) / max(len(all_tp) + len(all_fn), 1)
print(f'Precision: {precision:.4f}, Recall: {recall:.4f}')

## 2. Confusion Matrix Heatmap

In [ ]:
labels = CLASSES + ['background']
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(confusion, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Ground Truth', fontsize=12)
ax.set_title('Confusion Matrix — Test Set', fontsize=14)
plt.tight_layout()
plt.show()

## 3. False Positive Gallery

"They'll mostly be rock clusters" — let's verify.

In [ ]:
# Per-class FP breakdown
fp_by_class = defaultdict(list)
for img_path, pred, cls_id in all_fp:
    cls_name = CLASSES[cls_id] if cls_id < len(CLASSES) else f'cls_{cls_id}'
    fp_by_class[cls_name].append((img_path, pred))

print('False Positives by class:')
for cls_name in sorted(fp_by_class, key=lambda x: len(fp_by_class[x]), reverse=True):
    print(f'  {cls_name:20s}: {len(fp_by_class[cls_name])}')

# Bar chart
cls_names = sorted(fp_by_class.keys(), key=lambda x: len(fp_by_class[x]), reverse=True)
counts = [len(fp_by_class[c]) for c in cls_names]

plt.figure(figsize=(10, 4))
plt.barh(cls_names, counts, color='tomato', alpha=0.8)
plt.xlabel('Count')
plt.title('False Positives by Predicted Class')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# ---- Visual FP gallery ----
n_show = min(8, len(all_fp))
if n_show > 0:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    # Sort by confidence (highest-conf FPs are most instructive)
    sorted_fps = sorted(all_fp, key=lambda x: x[1]['conf'], reverse=True)
    
    for i in range(n_show):
        img_path, pred, cls_id = sorted_fps[i]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        x1, y1, x2, y2 = int(pred['x1']), int(pred['y1']), int(pred['x2']), int(pred['y2'])
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        
        cls_name = CLASSES[cls_id] if cls_id < len(CLASSES) else f'cls_{cls_id}'
        axes[i].imshow(img)
        axes[i].set_title(f'FP: {cls_name} ({pred["conf"]:.2f})', fontsize=10, color='red')
        axes[i].axis('off')
    
    for j in range(n_show, 8):
        axes[j].axis('off')
    
    plt.suptitle('Top-Confidence False Positives (red boxes)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('No false positives found!')

## 4. False Negative Gallery

In [ ]:
fn_by_class = defaultdict(list)
for img_path, gt, cls_id in all_fn:
    cls_name = CLASSES[cls_id] if cls_id < len(CLASSES) else f'cls_{cls_id}'
    fn_by_class[cls_name].append((img_path, gt))

print('False Negatives by class:')
for cls_name in sorted(fn_by_class, key=lambda x: len(fn_by_class[x]), reverse=True):
    print(f'  {cls_name:20s}: {len(fn_by_class[cls_name])}')

n_show = min(8, len(all_fn))
if n_show > 0:
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    for i in range(n_show):
        img_path, gt, cls_id = all_fn[i]
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        x1, y1, x2, y2 = int(gt['x1']), int(gt['y1']), int(gt['x2']), int(gt['y2'])
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
        
        cls_name = CLASSES[cls_id] if cls_id < len(CLASSES) else f'cls_{cls_id}'
        axes[i].imshow(img)
        axes[i].set_title(f'FN: {cls_name} (missed)', fontsize=10, color='green')
        axes[i].axis('off')
    
    for j in range(n_show, 8):
        axes[j].axis('off')
    
    plt.suptitle('False Negatives (green boxes = GT the model missed)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 5. Per-Class Error Summary

In [ ]:
tp_by_class = defaultdict(int)
for _, _, cls_id in all_tp:
    cls_name = CLASSES[cls_id] if cls_id < len(CLASSES) else f'cls_{cls_id}'
    tp_by_class[cls_name] += 1

print(f'{"Class":20s} {"TP":>5s} {"FP":>5s} {"FN":>5s} {"Precision":>10s} {"Recall":>8s}')
print('-' * 55)
for cls_name in CLASSES:
    tp = tp_by_class[cls_name]
    fp = len(fp_by_class.get(cls_name, []))
    fn = len(fn_by_class.get(cls_name, []))
    p = tp / max(tp + fp, 1)
    r = tp / max(tp + fn, 1)
    print(f'{cls_name:20s} {tp:5d} {fp:5d} {fn:5d} {p:10.4f} {r:8.4f}')

print()
print('→ Classes with high FP rates are candidates for shadow geometry filtering.')
print('→ Classes with high FN rates may need more training data or lower conf threshold.')

## 6. Next Steps

- **Rock-cluster FPs** → Shadow geometry filter (`06_shadow_geometry_filter.ipynb`)
- **Rare-class FNs** → More synthetic data + lower conf threshold
- **Class confusion** → Merge similar classes or add more training examples